# WAXAL ASR Challenge — First Submission (Zero-shot MMS baseline)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ashuza11/google-waxal-asr/blob/main/notebooks/01_zeroshot_mms_baseline.ipynb)

**Run this notebook on Google Colab with a GPU runtime**
(`Runtime > Change runtime type > T4 GPU`).

Confirmed against the real files downloaded from Zindi (`data/` in the
local repo — schema verified with `notebooks/00_data_exploration.py`):

- `Train.csv`: columns `id, transcription, language, original_split`
  (`original_split` is `train`/`validation` — this is the HF train+val
  splits merged). 38,199 rows: lin=16,244, sna=15,836, lug=6,119.
  **Quirk:** 23 rows have literal escaped quotes inside transcriptions
  (people quoting book/song titles) that break the default C CSV parser —
  read with `engine="python", escapechar="\\"`.
- `Test.csv`: **only one column, `ID`** — no language column. Language is
  recovered from the id prefix (`lin_9470` -> `lin`). 4,253 rows, 0 overlap
  with Train ids.
- `SampleSubmission.csv`: columns `ID, Target`, ids in the same order as
  `Test.csv`, `Target` placeholder is the literal string `"XXX"`.
- **Important, found by an actual failed submission**: Zindi's real
  validator checks against a combined id set of `Test.csv` (4,253 Phase 1
  ids) **and** `Test_phase2.csv` (1,500 Phase 2 ids, e.g. `ID_TBDTM`) —
  5,753 ids total — even though `SampleSubmission.csv` only lists the 4,253
  Phase 1 ones and Phase 2 audio isn't released yet. Submitting only the
  4,253 rows fails with `Missing entries for IDs ID_TBDTM, ...`. This
  notebook fills the Phase 2 ids with a `"."` placeholder so the file
  passes validation; those rows aren't scorable yet regardless.

What this notebook does:
1. You upload 4 CSVs (small, no need for programmatic Zindi download —
   the community `zindi` pip package isn't an official API, so we're not
   depending on it here; submission at the end is a manual upload).
2. Logs into Hugging Face (the WaxalNLP dataset card asks for this, though
   the dataset itself isn't gated) and loads the **test-split** audio for
   `lin`/`lug`/`sna` from `google/WaxalNLP`, joined to `Test.csv` by `id`.
3. Runs **zero-shot inference** with `facebook/mms-1b-all` — it ships a
   dedicated CTC adapter for each of `lin`/`sna`/`lug` (verified against
   the model repo file list), so no fine-tuning is needed for a first real
   score.
4. Sanity-checks WER/CER against `Train.csv` ground truth (streamed, not
   fully downloaded — the train+validation audio is ~10GB across all 3
   languages, too much to pull just for a sanity check).
5. Builds `submission.csv` shaped exactly like `SampleSubmission.csv`.

This is a **baseline** to get a first number on the board fast, not the
final model. Google's own starter notebook (`data/Waxal_Challenge_Starter_Code.ipynb`,
downloaded alongside the CSVs) fine-tunes `google/gemma-3n-E2B-it` with
LoRA per language — a stronger but much heavier next step (gated model,
A100 recommended, one language at a time). That's the natural "part two"
once this baseline is on the leaderboard.

In [ ]:
# ============================================================
# 0. Install packages
# ============================================================
get_ipython().system('apt-get -qq install -y ffmpeg > /dev/null')
get_ipython().system('pip install -q -U "transformers>=4.46" "datasets[audio]>=2.20" accelerate jiwer soundfile librosa pandas huggingface_hub')

In [ ]:
# ============================================================
# 1. Imports and config
# ============================================================
import re
import unicodedata
import random

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from jiwer import wer, cer
from transformers import AutoProcessor, Wav2Vec2ForCTC

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

LANGS = ["lin", "lug", "sna"]  # Lingala, Luganda, Shona
MMS_MODEL_ID = "facebook/mms-1b-all"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: no GPU detected. Runtime > Change runtime type > T4 GPU.")

## 2. Upload the challenge CSVs

Upload `Train.csv`, `Test.csv`, `SampleSubmission.csv`, **and
`Test_phase2.csv`** (from your local `data/` folder, downloaded from the
Zindi "Data" tab) — the last one is needed to satisfy the combined-id
validator, see the note above.

In [ ]:
import os
from google.colab import files

REQUIRED_FILES = ["Train.csv", "Test.csv", "SampleSubmission.csv", "Test_phase2.csv"]

while True:
    missing = [f for f in REQUIRED_FILES if not os.path.exists(f)]
    if not missing:
        break
    print(f"Still need: {missing} — select it/them in the picker (multi-select is fine).")
    files.upload()

print("All required files present:", REQUIRED_FILES)

In [ ]:
# ============================================================
# 3. Load the CSVs (see the escapechar note in the markdown above)
# ============================================================
train_df = pd.read_csv("Train.csv", engine="python", escapechar="\\")
test_df = pd.read_csv("Test.csv")
sample_sub_df = pd.read_csv("SampleSubmission.csv")
test_phase2_df = pd.read_csv("Test_phase2.csv")

print("Train.csv", train_df.shape)
display(train_df.head())
print("\nTest.csv", test_df.shape)
display(test_df.head())
print("\nSampleSubmission.csv", sample_sub_df.shape)
display(sample_sub_df.head())
print("\nTest_phase2.csv", test_phase2_df.shape)
display(test_phase2_df.head())

ID_COL_TRAIN = "id"
TEXT_COL = "transcription"
LANG_COL = "language"
ID_COL_TEST = "ID"
SUB_ID_COL, SUB_TEXT_COL = sample_sub_df.columns[:2]

# Test.csv has no language column — recover it from the id prefix.
test_df["language"] = test_df[ID_COL_TEST].str.split("_").str[0]
print("\nTest.csv language distribution (from id prefix):")
print(test_df["language"].value_counts())

assert set(test_df["language"].unique()) <= set(LANGS), "Unexpected language prefix in Test.csv ids"
assert sample_sub_df[SUB_ID_COL].tolist() == test_df[ID_COL_TEST].tolist(), \
    "SampleSubmission ids don't match Test.csv ids/order — check both files are the current Phase 1 versions"

## 4. Load WaxalNLP test-split audio and join by id

We only need the **test** split here (~1.3GB across all 3 languages) —
that's what `Test.csv` ids come from. Train+validation audio (~10GB) is
loaded via streaming in the sanity-check section instead of a full
download.

In [ ]:
def load_test_audio_index(lang: str) -> dict:
    ds = load_dataset("google/WaxalNLP", f"{lang}_asr", split="test")
    index = {row["id"]: row["audio"] for row in ds}
    print(f"{lang}: indexed {len(index)} test ids")
    return index

test_audio_index = {}
for lang in LANGS:
    test_audio_index.update(load_test_audio_index(lang))

print(f"\nTotal indexed test ids: {len(test_audio_index)}")
matched = test_df[ID_COL_TEST].isin(test_audio_index.keys()).sum()
print(f"Test.csv ids matched to WaxalNLP test audio: {matched}/{len(test_df)}")
if matched < len(test_df):
    missing_sample = test_df.loc[~test_df[ID_COL_TEST].isin(test_audio_index.keys()), ID_COL_TEST].head(5).tolist()
    print(f"Unmatched id examples: {missing_sample}")

## 5. Zero-shot MMS-1b-all baseline

`facebook/mms-1b-all` ships a dedicated CTC adapter per language
(`adapter.lin.*`, `adapter.lug.*`, `adapter.sna.*` all confirmed present in
the model repo). We swap the adapter per language and decode greedily —
no fine-tuning needed for a first real leaderboard score.

In [ ]:
processor = AutoProcessor.from_pretrained(MMS_MODEL_ID)
model = Wav2Vec2ForCTC.from_pretrained(MMS_MODEL_ID).to(DEVICE)
model.eval()

_current_adapter = None

def set_language(lang: str):
    global _current_adapter
    if _current_adapter != lang:
        processor.tokenizer.set_target_lang(lang)
        model.load_adapter(lang)
        _current_adapter = lang


@torch.no_grad()
def transcribe(audio_arrays: list, lang: str, batch_size: int = 8) -> list:
    set_language(lang)
    outputs = []
    for i in range(0, len(audio_arrays), batch_size):
        batch = audio_arrays[i:i + batch_size]
        inputs = processor(batch, sampling_rate=16_000, return_tensors="pt", padding=True)
        input_values = inputs.input_values.to(DEVICE)
        attn = inputs.get("attention_mask")
        if attn is not None:
            attn = attn.to(DEVICE)
        logits = model(input_values, attention_mask=attn).logits
        ids = torch.argmax(logits, dim=-1)
        outputs.extend(processor.batch_decode(ids))
    return outputs


def get_array(audio_field) -> np.ndarray:
    arr = np.asarray(audio_field["array"], dtype=np.float32)
    sr = audio_field.get("sampling_rate") or audio_field.get("sample_rate") or 16_000
    if sr != 16_000:
        import librosa
        arr = librosa.resample(arr, orig_sr=sr, target_sr=16_000)
    return arr


def normalise(text: str) -> str:
    text = "" if text is None else str(text)
    text = text.lower()
    text = unicodedata.normalize("NFD", text)
    text = "".join(c for c in text if unicodedata.category(c) != "Mn")
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def score(refs: list, hyps: list) -> dict:
    refs_n = [normalise(r) for r in refs]
    hyps_n = [normalise(h) for h in hyps]
    w = wer(refs_n, hyps_n)
    c = cer(refs_n, hyps_n)
    return {"wer": round(w, 4), "cer": round(c, 4), "score": round(0.5 * w + 0.5 * c, 4)}

## 6. Sanity check on a streamed slice of train+validation

Cheap gut-check before spending a submission: stream ~30 examples per
language (no full download) and report WER/CER/weighted score with the
same normalisation the leaderboard uses.

In [ ]:
N_SANITY_PER_LANG = 30

for lang in LANGS:
    stream = load_dataset("google/WaxalNLP", f"{lang}_asr", split="train", streaming=True)
    sample_rows = list(stream.shuffle(seed=SEED, buffer_size=1000).take(N_SANITY_PER_LANG))
    if not sample_rows:
        print(f"{lang}: no streamed rows, skipping")
        continue
    arrays = [get_array(r["audio"]) for r in sample_rows]
    refs = [r["transcription"] for r in sample_rows]
    hyps = transcribe(arrays, lang)
    result = score(refs, hyps)
    print(f"{lang}: n={len(sample_rows)}  WER={result['wer']:.3f}  CER={result['cer']:.3f}  score={result['score']:.3f}")
    for r, h in list(zip(refs, hyps))[:3]:
        print(f"    ref: {r}")
        print(f"    hyp: {h}")

## 7. Run inference on Test.csv and build submission.csv

In [ ]:
predictions = {}
for lang in LANGS:
    lang_rows = test_df[test_df["language"] == lang]
    ids = [i for i in lang_rows[ID_COL_TEST] if i in test_audio_index]
    if not ids:
        print(f"{lang}: no matched test rows")
        continue
    arrays = [get_array(test_audio_index[i]) for i in ids]
    hyps = transcribe(arrays, lang)
    for i, h in zip(ids, hyps):
        predictions[i] = normalise(h) or "."
    print(f"{lang}: transcribed {len(ids)} test rows")

missing = [i for i in test_df[ID_COL_TEST] if i not in predictions]
if missing:
    print(f"WARNING: {len(missing)} test ids had no matched audio — filling with '.' placeholder")
    for i in missing:
        predictions[i] = "."

## 7b. Add Phase 2 placeholder rows (required by Zindi's validator)

Phase 2 audio isn't released yet, so these can't have real predictions.
Zindi's submission validator checks against the combined Phase 1 + Phase 2
id set though (`Missing entries for IDs ID_TBDTM, ...` if you skip this),
so every `Test_phase2.csv` id needs *some* value here — a placeholder that
will simply score badly on those rows until Phase 2 opens, not one that
fails the format check.

In [ ]:
phase2_id_col = test_phase2_df.columns[0]
for i in test_phase2_df[phase2_id_col]:
    predictions[i] = "."

print(f"Total ids with a value: {len(predictions)} "
      f"(expected {len(test_df) + len(test_phase2_df)} = Test.csv + Test_phase2.csv)")

In [ ]:
submission = pd.concat(
    [sample_sub_df[[SUB_ID_COL]], test_phase2_df[[phase2_id_col]].rename(columns={phase2_id_col: SUB_ID_COL})],
    ignore_index=True,
)
submission[SUB_TEXT_COL] = submission[SUB_ID_COL].map(predictions)

assert submission[SUB_TEXT_COL].isna().sum() == 0, "Some submission rows have no prediction — check id matching."
assert len(submission) == len(sample_sub_df) + len(test_phase2_df), "Row count doesn't match Phase1 + Phase2 ids"

out_path = "submission_mms_zeroshot.csv"
submission.to_csv(out_path, index=False)
print(f"Saved {out_path}  shape={submission.shape}")
display(submission.head())
display(submission.tail())

files.download(out_path)

## 8. Submit to Zindi

`submission_mms_zeroshot.csv` just downloaded to your machine — it has all
5,753 ids (Phase 1 real predictions + Phase 2 placeholders). Upload it by
hand on the challenge's **Submissions** tab on Zindi — that's the reliable
path (no dependency on an unofficial API client, and it's just one file).
Costs one of your 5 daily / 200 total submissions, so check the
sanity-check scores above look sane first.